# post_train notebook

Tiny post-training playground for this nanoGPT implementation.

这个文件的目标不是做一个工业级 RLHF 系统，而是把后训练的核心链路
拆成最小可运行版本，方便边跑边理解：

1. SFT: 用 instruction/answer 数据继续做 next-token prediction。
   关键点是 prompt 只当上下文，answer 部分才计算 loss。
2. RLHS: 这里用 DPO 风格的偏好优化近似演示 RLHF/RLHS。
   关键点是让模型相对 reference model 更偏向 chosen，少偏向 rejected。

默认数据集直接写在本文件里，所以第一次理解流程时不用先处理数据下载。

这个 notebook 由 `post_train.py` 拆分而来：每个函数单独放在一个代码 cell 中，且每个代码 cell 前都有对应的 markdown 说明。


## 依赖、常量和 tiny 数据集

这一段负责导入训练需要的库，复用 `train_gpt2.py` 中的 `GPT/GPTConfig`，并定义最小可跑的 SFT 数据与偏好数据。

- `EOT` 是 GPT-2 tokenizer 的结束符。
- `SFT_EXAMPLES` 用于监督微调，格式是 `(instruction, answer)`。
- `PREFERENCE_EXAMPLES` 用于偏好训练，格式是 `(instruction, chosen, rejected)`。


In [ ]:
import argparse
import copy
import os
import random
from contextlib import nullcontext

# PyTorch 负责模型训练；F 里放了 softmax / cross_entropy / logsigmoid 等函数。
import torch
from torch.nn import functional as F

# tiktoken 使用 GPT-2 的 BPE tokenizer，和 train_gpt2.py 中的 token 空间一致。
import tiktoken

# 复用预训练文件里的模型定义，避免为后训练重复创造一套 GPT 接口。
from train_gpt2 import GPT, GPTConfig


# GPT-2 tokenizer 里的文本结束符。训练样本末尾加它，模型才知道回答到哪里结束。
EOT = "<|endoftext|>"


# SFT 数据格式：(instruction, answer)
# 这些例子故意很小，目标是跑通监督微调流程，不是训练出真正聪明的助手。
SFT_EXAMPLES = [
    ("What is 2 + 2?", "2 + 2 = 4."),
    ("Answer with one word: what color is the sky on a clear day?", "blue"),
    ("Rewrite politely: send me the file now.", "Please send me the file when you have a chance."),
    ("Give a tiny Python function that adds two numbers.", "def add(a, b):\n    return a + b"),
    ("What should a helpful assistant do when it is unsure?", "It should say it is unsure and ask for clarification."),
    ("Summarize this in five words: machine learning models learn patterns from data.", "Models learn patterns from data."),
]


# 偏好数据格式：(instruction, chosen, rejected)
# chosen 是希望模型更偏好的回答，rejected 是希望模型降低偏好的回答。
# 完整 RLHF 常见做法是先训练 reward model 再 PPO；这里用 DPO 风格目标，
# 可以直接从 chosen/rejected 对里学习，更适合最小教学版本。
PREFERENCE_EXAMPLES = [
    (
        "What is 2 + 2?",
        "2 + 2 = 4.",
        "2 + 2 = 5.",
    ),
    (
        "Answer with one word: what color is the sky on a clear day?",
        "blue",
        "banana",
    ),
    (
        "Rewrite politely: send me the file now.",
        "Please send me the file when you have a chance.",
        "Send it immediately.",
    ),
    (
        "Give a tiny Python function that adds two numbers.",
        "def add(a, b):\n    return a + b",
        "def add(a, b):\n    return a - b",
    ),
    (
        "What should a helpful assistant do when it is unsure?",
        "It should say it is unsure and ask for clarification.",
        "It should invent an answer.",
    ),
]


## `format_prompt`

统一 prompt 模板，让 SFT、偏好训练和采样看到同一种输入格式。

### 原始注释

把一条 instruction 包成固定模板。

模板很朴素，但它解决两个问题：
1. 让模型看到明确的“指令区”和“回答区”分界；
2. SFT/RLHS/采样都用同一种格式，避免训练和推理不一致。


In [ ]:
def format_prompt(instruction):
    """把一条 instruction 包成固定模板。

    模板很朴素，但它解决两个问题：
    1. 让模型看到明确的“指令区”和“回答区”分界；
    2. SFT/RLHS/采样都用同一种格式，避免训练和推理不一致。
    """
    return f"### Instruction:\n{instruction}\n\n### Response:\n"


## `encode`

把文本转为 GPT-2 token id，并允许 `<|endoftext|>` 作为特殊 token。

### 原始注释

把字符串转成 token id。

allowed_special={EOT} 表示允许把 <|endoftext|> 当作特殊 token 编码，
否则 tiktoken 可能会拒绝直接编码这个特殊字符串。


In [ ]:
def encode(enc, text):
    """把字符串转成 token id。

    allowed_special={EOT} 表示允许把 <|endoftext|> 当作特殊 token 编码，
    否则 tiktoken 可能会拒绝直接编码这个特殊字符串。
    """
    return enc.encode(text, allowed_special={EOT})


## `pick_device`

自动选择训练设备；显式传入 `--device` 时优先使用用户指定设备。

### 原始注释

选择训练设备。

用户显式传 --device 时尊重用户；否则按 cuda -> mps -> cpu 的顺序自动选择。


In [ ]:
def pick_device(requested):
    """选择训练设备。

    用户显式传 --device 时尊重用户；否则按 cuda -> mps -> cpu 的顺序自动选择。
    """
    if requested:
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


## `load_checkpoint`

兼容不同 PyTorch 版本读取 checkpoint。

### 原始注释

读取 checkpoint。

新版 PyTorch 的 torch.load 有 weights_only 参数；旧版没有。
这里用 try/except 同时兼容两类版本。


In [ ]:
def load_checkpoint(path):
    """读取 checkpoint。

    新版 PyTorch 的 torch.load 有 weights_only 参数；旧版没有。
    这里用 try/except 同时兼容两类版本。
    """
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


## `config_from_checkpoint`

把 checkpoint 中保存的 config 恢复成 `GPTConfig`。

### 原始注释

从 checkpoint 里恢复 GPTConfig。

train_gpt2.py 保存的 config 可能是 dataclass 对象，也可能在其他实验里被存成 dict。
这个函数把不同形态统一转回 GPTConfig。


In [ ]:
def config_from_checkpoint(checkpoint):
    """从 checkpoint 里恢复 GPTConfig。

    train_gpt2.py 保存的 config 可能是 dataclass 对象，也可能在其他实验里被存成 dict。
    这个函数把不同形态统一转回 GPTConfig。
    """
    config = checkpoint.get("config")
    if config is None:
        raise ValueError("checkpoint does not contain a 'config' entry")
    if isinstance(config, GPTConfig):
        return config
    if isinstance(config, dict):
        return GPTConfig(**config)
    return GPTConfig(**vars(config))


## `strip_state_dict_prefixes`

处理 DDP 或 torch.compile 给参数名添加的前缀，保证普通模型可以加载权重。

### 原始注释

去掉分布式训练或 torch.compile 可能加上的参数名前缀。

DDP 常见前缀是 module.，torch.compile 常见前缀是 _orig_mod.。
去掉以后，普通 GPT(...) 才能直接 load_state_dict。


In [ ]:
def strip_state_dict_prefixes(state_dict):
    """去掉分布式训练或 torch.compile 可能加上的参数名前缀。

    DDP 常见前缀是 module.，torch.compile 常见前缀是 _orig_mod.。
    去掉以后，普通 GPT(...) 才能直接 load_state_dict。
    """
    for prefix in ("module.", "_orig_mod."):
        if state_dict and all(key.startswith(prefix) for key in state_dict):
            state_dict = {key[len(prefix):]: value for key, value in state_dict.items()}
    return state_dict


## `create_model`

从 checkpoint 恢复模型，或创建一个 CPU 也能跑的 tiny scratch GPT。

### 原始注释

创建后训练模型。

两种入口：
- 传 --init-from：从已有 checkpoint 继续后训练；
- 不传 --init-from：创建一个很小的 scratch 模型，用来快速验证代码能跑通。


In [ ]:
def create_model(args, device):
    """创建后训练模型。

    两种入口：
    - 传 --init-from：从已有 checkpoint 继续后训练；
    - 不传 --init-from：创建一个很小的 scratch 模型，用来快速验证代码能跑通。
    """
    if args.init_from:
        checkpoint = load_checkpoint(args.init_from)
        config = config_from_checkpoint(checkpoint)
        state_dict = checkpoint.get("model", checkpoint)
        state_dict = strip_state_dict_prefixes(state_dict)
        model = GPT(config)
        model.load_state_dict(state_dict)
        print(f"loaded checkpoint: {args.init_from}")
    else:
        # scratch tiny model 的默认规模很小，CPU 上也能跑。
        # 如果想接 train_gpt2.py 的大模型，应该用 --init-from 加载 checkpoint。
        config = GPTConfig(
            block_size=args.block_size,
            vocab_size=50304,
            n_layer=args.n_layer,
            n_head=args.n_head,
            n_embd=args.n_embd,
        )
        model = GPT(config)
        print("initialized a tiny GPT from scratch")
    model.to(device)
    return model


## `save_checkpoint`

保存模型权重、config、训练阶段和 step，方便后续继续训练。

### 原始注释

保存后训练 checkpoint。

保存 model 权重和 config，之后可以继续接着做 SFT/RLHS 或单独加载采样。


In [ ]:
def save_checkpoint(path, model, stage, step):
    """保存后训练 checkpoint。

    保存 model 权重和 config，之后可以继续接着做 SFT/RLHS 或单独加载采样。
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save(
        {
            "model": model.state_dict(),
            "config": model.config,
            "stage": stage,
            "step": step,
        },
        path,
    )
    print(f"saved {stage} checkpoint: {path}")


## `build_sft_batch`

构造 SFT batch。核心是 next-token shift，以及把 prompt 区域 label 设为 `-100` 来忽略 loss。

### 原始注释

构造一个 SFT mini-batch。

GPT 训练的基本形式是：
x = 前 T 个 token
y = 后 T 个 token
模型看到 x 的每个位置，去预测 y 的对应位置。

对 instruction tuning 来说，我们只希望模型“学习怎么回答”，
不希望它把 prompt 本身也当成需要模仿的答案，所以 prompt 区域的 y 会被设为 -100。
PyTorch cross_entropy 默认 ignore_index=-100，这些位置不会贡献 loss。


In [ ]:
def build_sft_batch(rows, enc, block_size, batch_size, device):
    """构造一个 SFT mini-batch。

    GPT 训练的基本形式是：
    x = 前 T 个 token
    y = 后 T 个 token
    模型看到 x 的每个位置，去预测 y 的对应位置。

    对 instruction tuning 来说，我们只希望模型“学习怎么回答”，
    不希望它把 prompt 本身也当成需要模仿的答案，所以 prompt 区域的 y 会被设为 -100。
    PyTorch cross_entropy 默认 ignore_index=-100，这些位置不会贡献 loss。
    """
    pad_id = enc.eot_token

    # 简单起见，每一步从内置小数据里有放回随机采样 batch_size 条。
    chosen = [random.choice(rows) for _ in range(batch_size)]
    encoded_rows = []
    max_t = 0
    for instruction, answer in chosen:
        # prompt 是上下文，answer 是我们希望模型学会生成的目标文本。
        prompt_ids = encode(enc, format_prompt(instruction))
        answer_ids = encode(enc, answer + EOT)

        # 多取 1 个 token 是为了做 next-token shift：
        # tokens[:-1] 给模型看，tokens[1:] 作为每个位置要预测的目标。
        tokens = (prompt_ids + answer_ids)[: block_size + 1]
        if len(tokens) < 2:
            raise ValueError("encoded example is too short")
        x = tokens[:-1]
        y = tokens[1:]

        # y 中 prompt 对应的位置不算 loss。
        # len(prompt_ids) - 1 是因为 y 已经相对 x 左移了一格：
        # x 的最后一个 prompt token 应该预测第一个 answer token，这个位置需要保留 loss。
        ignore_until = min(max(len(prompt_ids) - 1, 0), len(y))
        y = [-100] * ignore_until + y[ignore_until:]
        if all(token == -100 for token in y):
            raise ValueError("block_size is too small; no answer tokens are left for SFT loss")
        encoded_rows.append((x, y))
        max_t = max(max_t, len(x))

    # 同一个 batch 里的序列长度要一致；短样本用 EOT padding。
    # padding 的 label 设为 -100，避免模型被训练去预测 padding。
    xs, ys = [], []
    for x, y in encoded_rows:
        pad_len = max_t - len(x)
        xs.append(x + [pad_id] * pad_len)
        ys.append(y + [-100] * pad_len)
    return (
        torch.tensor(xs, dtype=torch.long, device=device),
        torch.tensor(ys, dtype=torch.long, device=device),
    )


## `build_completion_tensors`

构造 chosen/rejected completion 的 token、target 和 mask，用于计算 completion log probability。

### 原始注释

把若干 completion 编码成 DPO/RLHS 需要的张量。

与 SFT 类似，这里也有 x/y shift。
不同点是：DPO 需要计算每个 completion 的总 log probability，
所以我们用 mask 标记哪些 token 属于 completion，prompt token 不计入分数。


In [ ]:
def build_completion_tensors(rows, enc, block_size, device):
    """把若干 completion 编码成 DPO/RLHS 需要的张量。

    与 SFT 类似，这里也有 x/y shift。
    不同点是：DPO 需要计算每个 completion 的总 log probability，
    所以我们用 mask 标记哪些 token 属于 completion，prompt token 不计入分数。
    """
    pad_id = enc.eot_token
    encoded_rows = []
    max_t = 0
    for instruction, completion in rows:
        prompt_ids = encode(enc, format_prompt(instruction))
        completion_ids = encode(enc, completion + EOT)
        tokens = (prompt_ids + completion_ids)[: block_size + 1]
        if len(tokens) < 2:
            raise ValueError("encoded example is too short")
        x = tokens[:-1]
        y = tokens[1:]

        # mask=1 的位置会被计入 completion log probability；
        # mask=0 的位置只是上下文，不参与 chosen/rejected 打分。
        mask_start = min(max(len(prompt_ids) - 1, 0), len(y))
        mask = [0.0] * mask_start + [1.0] * (len(y) - mask_start)
        if sum(mask) == 0:
            raise ValueError("block_size is too small; no completion tokens are left for RLHS loss")
        encoded_rows.append((x, y, mask))
        max_t = max(max_t, len(x))

    # padding 后 mask 补 0，表示 padding 不参与 log probability 求和。
    xs, ys, masks = [], [], []
    for x, y, mask in encoded_rows:
        pad_len = max_t - len(x)
        xs.append(x + [pad_id] * pad_len)
        ys.append(y + [pad_id] * pad_len)
        masks.append(mask + [0.0] * pad_len)
    return (
        torch.tensor(xs, dtype=torch.long, device=device),
        torch.tensor(ys, dtype=torch.long, device=device),
        torch.tensor(masks, dtype=torch.float32, device=device),
    )


## `build_preference_batch`

从偏好数据中采样 batch，并拆成 chosen 与 rejected 两组张量。

### 原始注释

构造偏好训练 batch。

返回两组张量：
- chosen:   人类/规则更偏好的回答
- rejected: 不希望模型偏好的回答
DPO loss 会比较两组回答在 policy model 与 reference model 下的相对分数。


In [ ]:
def build_preference_batch(rows, enc, block_size, batch_size, device):
    """构造偏好训练 batch。

    返回两组张量：
    - chosen:   人类/规则更偏好的回答
    - rejected: 不希望模型偏好的回答
    DPO loss 会比较两组回答在 policy model 与 reference model 下的相对分数。
    """
    batch = [random.choice(rows) for _ in range(batch_size)]
    chosen_rows = [(instruction, chosen) for instruction, chosen, _ in batch]
    rejected_rows = [(instruction, rejected) for instruction, _, rejected in batch]
    return (
        build_completion_tensors(chosen_rows, enc, block_size, device),
        build_completion_tensors(rejected_rows, enc, block_size, device),
    )


## `sequence_logps`

计算每条 completion 的 log probability 总和，这是 DPO/RLHS 比较 chosen/rejected 的基础分数。

### 原始注释

计算每条序列中 completion 部分的 log probability 总和。

logits 形状是 (B, T, vocab_size)。
log_softmax 后，每个位置都有整个词表的 log probability。
gather 会把 y 指定的“真实下一个 token”的 log probability 取出来。
最后乘 mask，只保留 answer/completion 区域。


In [ ]:
def sequence_logps(model, x, y, mask):
    """计算每条序列中 completion 部分的 log probability 总和。

    logits 形状是 (B, T, vocab_size)。
    log_softmax 后，每个位置都有整个词表的 log probability。
    gather 会把 y 指定的“真实下一个 token”的 log probability 取出来。
    最后乘 mask，只保留 answer/completion 区域。
    """
    logits, _ = model(x)
    log_probs = F.log_softmax(logits, dim=-1)
    token_logps = torch.gather(log_probs, dim=-1, index=y.unsqueeze(-1)).squeeze(-1)
    return (token_logps * mask).sum(dim=-1)


## `optimizer_for`

复用原训练脚本的 AdamW 参数分组逻辑。

### 原始注释

创建优化器。

这里复用 train_gpt2.py 里的 configure_optimizers，
保持 weight decay 分组和 AdamW 细节与原训练脚本一致。


In [ ]:
def optimizer_for(model, args, device):
    """创建优化器。

    这里复用 train_gpt2.py 里的 configure_optimizers，
    保持 weight decay 分组和 AdamW 细节与原训练脚本一致。
    """
    device_type = "cuda" if str(device).startswith("cuda") else "cpu"
    return model.configure_optimizers(
        weight_decay=args.weight_decay,
        learning_rate=args.learning_rate,
        device_type=device_type,
    )


## `autocast_context`

CUDA 上启用 bfloat16 自动混合精度；其他设备保持普通精度。

### 原始注释

自动混合精度上下文。

CUDA 上用 bfloat16 加速；CPU/MPS 先用普通精度，减少兼容问题。
nullcontext() 的意思是“什么都不额外做”，但可以和 with 语句统一写法。


In [ ]:
def autocast_context(device):
    """自动混合精度上下文。

    CUDA 上用 bfloat16 加速；CPU/MPS 先用普通精度，减少兼容问题。
    nullcontext() 的意思是“什么都不额外做”，但可以和 with 语句统一写法。
    """
    if str(device).startswith("cuda"):
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


## `train_sft`

执行监督微调。SFT 仍然是语言模型 next-token prediction，只是 loss 只看 answer 区域。

### 原始注释

执行监督微调 SFT。

SFT 本质上仍然是语言模型训练：
给模型看 prompt + answer 的前半段，让它预测下一个 token。
只是我们在 build_sft_batch 里把 prompt 区域的 label 设成 -100，
所以 loss 只来自 answer 区域。


In [ ]:
def train_sft(args, device, out_path):
    """执行监督微调 SFT。

    SFT 本质上仍然是语言模型训练：
    给模型看 prompt + answer 的前半段，让它预测下一个 token。
    只是我们在 build_sft_batch 里把 prompt 区域的 label 设成 -100，
    所以 loss 只来自 answer 区域。
    """
    enc = tiktoken.get_encoding("gpt2")
    model = create_model(args, device)
    optimizer = optimizer_for(model, args, device)
    model.train()

    for step in range(args.steps):
        # 1. 从 tiny SFT 数据中采样并编码 batch。
        x, y = build_sft_batch(SFT_EXAMPLES, enc, model.config.block_size, args.batch_size, device)

        # 2. 标准 PyTorch 训练三步：清梯度 -> forward/loss -> backward/update。
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device):
            _, loss = model(x, y)
        loss.backward()

        # 3. 梯度裁剪防止偶发梯度过大，把训练炸掉。
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
        optimizer.step()
        if step % args.log_interval == 0 or step == args.steps - 1:
            print(f"sft step {step:04d} | loss {loss.item():.4f} | norm {norm:.4f}")

    # 训练结束后保存 checkpoint，并用一个 prompt 简单采样看看模型能不能生成。
    save_checkpoint(out_path, model, "sft", args.steps)
    sample(model, enc, args.sample_prompt, device, args.max_new_tokens)
    return out_path


## `train_rlhs`

执行最小 DPO 风格偏好训练。reference model 冻结，policy model 学会更偏向 chosen。

### 原始注释

执行最小偏好后训练 RLHS。

这里的 RLHS 使用 DPO 风格目标：
- policy model: 当前正在训练的模型
- reference model: 训练开始时的冻结副本
- chosen/rejected: 同一个 prompt 下，好回答和差回答

目标不是单纯让 chosen 概率变大，而是让 policy 相对于 reference
更偏向 chosen、少偏向 rejected。reference model 像一个“不要偏离太远”的锚点。


In [ ]:
def train_rlhs(args, device, out_path):
    """执行最小偏好后训练 RLHS。

    这里的 RLHS 使用 DPO 风格目标：
    - policy model: 当前正在训练的模型
    - reference model: 训练开始时的冻结副本
    - chosen/rejected: 同一个 prompt 下，好回答和差回答

    目标不是单纯让 chosen 概率变大，而是让 policy 相对于 reference
    更偏向 chosen、少偏向 rejected。reference model 像一个“不要偏离太远”的锚点。
    """
    enc = tiktoken.get_encoding("gpt2")
    model = create_model(args, device)

    # reference_model 是当前模型的冻结拷贝。
    # 它不更新，只用于衡量“训练后的模型相对原模型改变了多少偏好”。
    reference_model = copy.deepcopy(model).eval()
    for param in reference_model.parameters():
        param.requires_grad = False

    optimizer = optimizer_for(model, args, device)
    model.train()

    for step in range(args.steps):
        # cx/cy/cmask 对应 chosen；rx/ry/rmask 对应 rejected。
        # c/r 是 chosen/rejected 的缩写。
        (cx, cy, cmask), (rx, ry, rmask) = build_preference_batch(
            PREFERENCE_EXAMPLES, enc, model.config.block_size, args.batch_size, device
        )
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device):
            # policy model 对 chosen/rejected completion 的 log probability。
            pi_chosen = sequence_logps(model, cx, cy, cmask)
            pi_rejected = sequence_logps(model, rx, ry, rmask)

            # reference model 不训练，所以这里 no_grad，节省显存/内存和计算图。
            with torch.no_grad():
                ref_chosen = sequence_logps(reference_model, cx, cy, cmask)
                ref_rejected = sequence_logps(reference_model, rx, ry, rmask)

            # advantage 表示 policy 相对 reference 给某个回答提高了多少 log probability。
            chosen_advantage = pi_chosen - ref_chosen
            rejected_advantage = pi_rejected - ref_rejected

            # DPO 核心：希望 chosen_advantage > rejected_advantage。
            # beta 控制偏好优化强度；越大，模型越激进地拉开 chosen/rejected。
            # -logsigmoid(...) 是一个二分类式 loss：
            # 当 chosen 比 rejected 分数高很多时，loss 接近 0。
            loss = -F.logsigmoid(args.beta * (chosen_advantage - rejected_advantage)).mean()
        loss.backward()
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
        optimizer.step()
        if step % args.log_interval == 0 or step == args.steps - 1:
            # margin 越大，表示模型相对 reference 越偏向 chosen。
            margin = (chosen_advantage - rejected_advantage).mean().item()
            print(f"rlhs step {step:04d} | loss {loss.item():.4f} | margin {margin:.4f} | norm {norm:.4f}")

    save_checkpoint(out_path, model, "rlhs", args.steps)
    sample(model, enc, args.sample_prompt, device, args.max_new_tokens)
    return out_path


## `sample`

训练后做一个轻量 sanity check，确认模型可以正常生成。

### 原始注释

用当前模型做一个简单采样。

这里只是 sanity check：确认训练后的 checkpoint 能正常 forward 和生成。
它不是严肃评测，所以没有 beam search、temperature、top-k 等完整采样控制。


In [ ]:
@torch.no_grad()
def sample(model, enc, prompt, device, max_new_tokens):
    """用当前模型做一个简单采样。

    这里只是 sanity check：确认训练后的 checkpoint 能正常 forward 和生成。
    它不是严肃评测，所以没有 beam search、temperature、top-k 等完整采样控制。
    """
    model.eval()

    # 采样时也使用训练同款 prompt 模板，保持格式一致。
    ids = encode(enc, format_prompt(prompt))
    x = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        # 如果上下文超过模型 block_size，只保留最后 block_size 个 token。
        x_cond = x[:, -model.config.block_size :]
        logits, _ = model(x_cond)

        # 只取最后一个位置的 logits，因为我们只需要预测“下一个 token”。
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)

        # multinomial 按概率随机抽样一个 token。它比 argmax 更能暴露模型会怎么生成。
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat((x, next_id), dim=1)
        if next_id.item() == enc.eot_token:
            break
    print("sample:")
    print(enc.decode(x[0].tolist()))

    # 恢复 train 模式，避免调用 sample 后影响后续训练。
    model.train()


## `parse_args`

定义脚本版命令行参数。Notebook 中通常不直接调用它，除非你确实想模拟命令行运行。

### 原始注释

解析命令行参数。

常用方式：
- python3 post_train.py --stage all
- python3 post_train.py --stage sft --init-from log/model_19072.pt --out log/sft.pt
- python3 post_train.py --stage rlhs --init-from log/sft.pt --out log/rlhs.pt


In [ ]:
def parse_args():
    """解析命令行参数。

    常用方式：
    - python3 post_train.py --stage all
    - python3 post_train.py --stage sft --init-from log/model_19072.pt --out log/sft.pt
    - python3 post_train.py --stage rlhs --init-from log/sft.pt --out log/rlhs.pt
    """
    parser = argparse.ArgumentParser(description="Tiny SFT and RLHS/DPO post-training for build-nanogpt.")

    # 选择跑哪个阶段：只跑 SFT、只跑 RLHS，或者先 SFT 再 RLHS。
    parser.add_argument("--stage", choices=["sft", "rlhs", "all"], default="all")

    # checkpoint 输入/输出。--out 用于单阶段；--sft-out/--rlhs-out 用于 --stage all。
    parser.add_argument("--init-from", type=str, default=None, help="optional checkpoint from train_gpt2.py or this script")
    parser.add_argument("--out", type=str, default=None, help="output checkpoint path for a single stage")
    parser.add_argument("--sft-out", type=str, default="log/sft_tiny.pt")
    parser.add_argument("--rlhs-out", type=str, default="log/rlhs_tiny.pt")

    # 训练超参数。默认值偏小，优先保证 CPU/Mac 上可以快速跑通。
    parser.add_argument("--steps", type=int, default=20)
    parser.add_argument("--rlhs-steps", type=int, default=None)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--learning-rate", type=float, default=3e-4)
    parser.add_argument("--weight-decay", type=float, default=0.01)
    parser.add_argument("--grad-clip", type=float, default=1.0)
    parser.add_argument("--beta", type=float, default=0.1, help="DPO/RLHS preference strength")
    parser.add_argument("--device", type=str, default=None, help="cpu, cuda, cuda:0, or mps")
    parser.add_argument("--seed", type=int, default=1337)
    parser.add_argument("--log-interval", type=int, default=5)

    # 训练结束后用于 sanity-check 采样的 prompt 和生成长度。
    parser.add_argument("--sample-prompt", type=str, default="What is 2 + 2?")
    parser.add_argument("--max-new-tokens", type=int, default=32)

    # 不传 --init-from 时，会创建 scratch tiny model；下面参数只影响这个 tiny model。
    # 如果从 train_gpt2.py checkpoint 加载，模型结构来自 checkpoint 中保存的 config。
    parser.add_argument("--block-size", type=int, default=96, help="scratch tiny model context length")
    parser.add_argument("--n-layer", type=int, default=2, help="scratch tiny model layers")
    parser.add_argument("--n-head", type=int, default=2, help="scratch tiny model heads")
    parser.add_argument("--n-embd", type=int, default=64, help="scratch tiny model embedding size")
    return parser.parse_args()


## `main`

脚本入口，根据 `--stage` 分发到 SFT、RLHS 或串联流程。Notebook 中建议手动调用训练函数。

### 原始注释

脚本入口：设随机种子、选设备，然后按 stage 分发到对应训练函数。


In [ ]:
def main():
    """脚本入口：设随机种子、选设备，然后按 stage 分发到对应训练函数。"""
    args = parse_args()

    # 固定随机种子，让 tiny 示例每次运行更容易复现。
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)
    device = pick_device(args.device)
    print(f"using device: {device}")

    if args.stage == "sft":
        # 单独跑 SFT：可以从 scratch tiny model 开始，也可以 --init-from 接预训练 checkpoint。
        train_sft(args, device, args.out or args.sft_out)
    elif args.stage == "rlhs":
        # 单独跑 RLHS：通常建议 --init-from 接 SFT 后的 checkpoint。
        train_rlhs(args, device, args.out or args.rlhs_out)
    else:
        # 串联跑：先保存 SFT checkpoint，再把它作为 RLHS 的初始化。
        sft_path = train_sft(args, device, args.sft_out)
        rlhs_args = copy.copy(args)
        rlhs_args.init_from = sft_path

        # 允许 SFT 和 RLHS 使用不同步数；没传 --rlhs-steps 时，两阶段都用 --steps。
        if args.rlhs_steps is not None:
            rlhs_args.steps = args.rlhs_steps
        train_rlhs(rlhs_args, device, args.rlhs_out)


## Notebook 运行提示

脚本文件最后的 `if __name__ == "__main__": main()` 没有放进可执行代码 cell。Notebook 中如果全量运行它，`argparse` 可能会读取 Jupyter 自带参数而报错。

建议在 notebook 中直接调用 `train_sft(...)`、`train_rlhs(...)`，或者在终端继续运行 `python3 post_train.py --stage all`。
